In [ ]:
%load_ext autoreload
%autoreload 2
import os
import json
import torch
import pycolmap
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
import torchvision.transforms as T

In [ ]:
dataset = "terrasky3D" # terrasky3D, mipnerf360, scannetpp
scene = "graz_townhall" # vienna_state_opera, bicycle, bonsai, graz_townhall, graz_church, 40aec5fffa

# Load dataset paths and parameters from JSON
with open("benchmarks/paths.json") as f:
    paths_cfg = json.load(f)

dataset_cfg = paths_cfg[dataset]

images_path = os.path.join(
    dataset_cfg["images_path"], scene, dataset_cfg["images_folder"]
)
base_path = dataset_cfg["base_path"]
reconstruction_path = os.path.join(
    base_path, scene, dataset_cfg["reconstruction_folder"]
)
depths_path = os.path.join(
    base_path,
    scene,
    dataset_cfg.get("depths_folder", dataset_cfg.get("depth_folder", "")),
)
gt_path = os.path.join(dataset_cfg["gt_path"], scene, dataset_cfg["gt_folder"])
opt = f"optimized_reconstruction_GD/{scene}"

print("Images path:", images_path)
print("Reconstruction path:", reconstruction_path)
print("Depths path:", depths_path)
print("GT path:", gt_path)

In [ ]:
from epo import EPO

epo = EPO(
    reconstruction_path = reconstruction_path,
    images_path = images_path,
    depths_path = depths_path,
    grad_k=True,
    single_camera_per_folder=True,
    detector="canny",  # or "canny", "bdcn", "sam2", "diff"
    detector_params={
        "low_threshold": 0.15,
        "high_threshold": 0.20,
        "kernel_size": 9,
        "sigma": 2,
    },
    verbose=True,
    max_edges_points=1024*12,
    max_viewgraph_pairs=1024*4,
    max_num_iterations=2000,
    seed=42,

    use_amp=True,  
    backend="triton",  
)

In [ ]:
from mylib.plot import plot_imgs

keys = sorted(list(epo.images.keys()))
print(epo.images[keys[0]].keys())

for i in range(0,150, 15):
    k = keys[i]
    rgb = epo.images[k]['image'].permute(1,2,0).cpu() if 'image' in epo.images[k] else torch.zeros_like(epo.images[k]['depth']).permute(1,2,0).cpu()
    depth = epo.images[k]['depth'].cpu() if 'depth' in epo.images[k] else torch.zeros_like(epo.images[k]['depth']).cpu()
    edges_map = epo.images[k]['edges_map'].cpu() if 'edges_map' in epo.images[k] else torch.zeros_like(epo.images[k]['depth']).cpu()
    # conf = epo.images[k]['confidence'].cpu() if 'confidence' in epo.images[k] else torch.zeros_like(epo.images[k]['depth']).cpu()
    # conf = (conf - conf.min()) / (conf.max() - conf.min() + 1e-8)
    dt_field = epo.images[k]['dt_field'].cpu() if 'dt_field' in epo.images[k] else torch.zeros_like(epo.images[k]['depth']).cpu()
    plot_imgs([rgb, depth, edges_map, dt_field], titles=[k, "Depth", "Edge Map", "DT Field"], figsize=(15,5))
    break

In [ ]:
epo(
    early_stop="pose",
    # gt_path=gt_path, 
)

In [ ]:
loss = epo.loss_list
lr_list = epo.lr_list
auc = epo.auc_list['auc']
print("Max AUC@5:", np.max(auc[5])) if len(auc[5]) > 0 else 0.0
# plot loss and lr side by side
n_plots = 4
plt.figure(figsize=(n_plots*6,3))
plt.subplot(1, n_plots, 1)
plt.plot(loss, label='Loss')
plt.legend()
plt.subplot(1, n_plots, 2)
for group, lr in lr_list.items():
    lr = np.array(lr)
    if len(lr) > 0:
        steps, lrs = lr[:,0],lr[:,1]
        plt.plot(steps, lrs, label=f'LR of {group}')
plt.legend()
plt.subplot(1, n_plots, 3)
for th in [1,3,5]: #epo.auc_th:
    plt.plot(epo.auc_list["steps"], auc[th], label=f'AUC@{th}px')
plt.legend() if len(epo.auc_list) > 0 else None
plt.subplot(1, n_plots, 4)
plt.plot(epo.changes["steps"], epo.changes["q"], label='Rotation change (deg)')
plt.plot(epo.changes["steps"], epo.changes["t"], label='Translation change (m)')
plt.plot(epo.changes["steps"], epo.changes["max"], label='Max change')
plt.xlabel('Optimization steps')
plt.ylabel('Change')
plt.legend()
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import json

# Parameters
window1 = 25
window2 = 50
window3 = 25 # loss
tol1 = 0.5
tol2 = 0.1 # 0.08, 0.07
tol3 = 5e-4 # loss

steps = np.array(epo.changes["steps"])
max_changes = np.array([x.item() if torch.is_tensor(x) else x for x in epo.changes["max"]])
auc = epo.auc_list['auc']
raw_loss = np.array(epo.loss_list)

# # 1. CLEAN DATA
# path = "benchmarks/vggt_edge_canny_final_no_stop/mipnerf360"
# scene = "flowers"
# data = json.load(open(f"benchmarks/vggt_edge_canny_final_no_stop/mipnerf360/{scene}/sparse/training_logs.json"))
# steps = np.arange(data["steps_actual"])
# max_changes = np.array([x.item() if torch.is_tensor(x) else x for x in data['list_changes']['max']])
# auc = data['list_auc']['auc']
# auc = {int(k):v for k,v in auc.items()}
# raw_loss = np.array(data['list_loss'])


if True:
    fig, ax1 = plt.subplots(figsize=(14, 4))

    # --- Primary Axis: Changes ---
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Max Change (Delta)', color='blue')
    
    # Plot Raw Data
    ax1.plot(steps, max_changes, alpha=0.15, label='Raw Max Change', color='blue', zorder=1)

    # Plot Smoothed Line for Window 1
    if len(max_changes) >= window1:
        smoothed1 = np.convolve(max_changes, np.ones(window1)/window1, mode='valid')
        # Offset steps to align with the end of the rolling window
        ax1.plot(steps[window1-1:], smoothed1, label=f'Smoothed (W={window1})', color='darkblue', linewidth=2, zorder=3)

    # Plot Smoothed Line for Window 2 (Visualizing what Tol2 sees)
    if len(max_changes) >= window2:
        smoothed2 = np.convolve(max_changes, np.ones(window2)/window2, mode='valid')
        ax1.plot(steps[window2-1:], smoothed2, label=f'Smoothed (W={window2})', color='teal', linewidth=1.5, linestyle='--', zorder=3)
    

    # Horizontal Tolerance Lines
    ax1.axhline(y=tol1, color='red', linestyle=':', alpha=0.6, label=f'Tol {tol1}')
    ax1.axhline(y=tol2, color='green', linestyle=':', alpha=0.6, label=f'Tol {tol2}')

    # --- Convergence Detection ---
    # We calculate convergence at each step based on the preceding window
    conv1 = [epo.check_convergence(max_changes[:i+1], "pose", window1, tol1) for i in range(len(max_changes))]
    conv2 = [epo.check_convergence(max_changes[:i+1], "pose", window2, tol2) for i in range(len(max_changes))]
    conv_loss = [epo.check_convergence(raw_loss[:i+1], "loss", window3, tol3) for i in range(len(raw_loss))]

    def plot_conv_line(conv_array, color, label_text):
        has_labeled = False
        limit_y = max(max_changes) if len(max_changes) > 0 else 1.0
        for c in range(1, len(conv_array)):
            # Detect the rising edge (False -> True)
            if conv_array[c] and not conv_array[c - 1]:
                label = label_text if not has_labeled else None
                ax1.vlines(steps[c], 0, limit_y, color=color, linestyle='--', linewidth=2, label=label, zorder=4)
                has_labeled = True

    plot_conv_line(conv1, 'red', f'Reached Tol {tol1}')
    plot_conv_line(conv2, 'green', f'Reached Tol {tol2}')

    # Scale loss for plotting (visual only) relative to max_changes range
    if len(raw_loss) > 0:
        # Use steps[:len(raw_loss)] to ensure X-axis alignment if lengths differ
        loss_x = steps[:len(raw_loss)] 
        l_min, l_max = raw_loss.min(), raw_loss.max()
        m_min, m_max = max_changes.min(), max_changes.max()
        
        # Avoid division by zero if loss is constant
        denom = (l_max - l_min) if l_max != l_min else 1
        scaled_loss = (raw_loss - l_min) / denom * (m_max - m_min) + m_min
        
        ax1.plot(loss_x, scaled_loss, label='Scaled Loss Trend', color='purple', alpha=0.5, linestyle=':')

    plot_conv_line(conv_loss, 'black', f'Loss converged (<{tol3})')

    
    # --- Secondary Axis: AUC ---
    if 5 in auc and len(auc[5]) > 0:
        ax2 = ax1.twinx()
        ax2.set_ylabel('AUC@5', color='orange')
        
        # Correctly align AUC steps based on saving frequency
        auc_steps = np.arange(len(auc[5])) * epo.auc_saving_freq
        ax2.plot(auc_steps, auc[5], label='AUC@5', color='orange', marker='o', markersize=4, alpha=0.8)
        
        # Annotate AUC values
        for i, (x, y) in enumerate(zip(auc_steps, auc[5])):
            c = 0.5 if i%2==0 else -1.5
            ax2.text(x, y+c, f'{y:.2f}', color='black', ha='center', va='bottom', fontsize=8, fontweight='bold')
            
        # Dynamic Y-limits for AUC
        y_min, y_max = min(auc[5]), max(auc[5])
        ax2.set_ylim(y_min - 0.02, y_max*1.02)

        # Merge Legends from both axes
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize='small', ncol=2, bbox_to_anchor=(1.05, 1))
    else:
        ax1.legend(loc='upper left', fontsize='small', ncol=2, bbox_to_anchor=(1.05, 1))

    ax1.set_title("Training Convergence & Stability Metrics")
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
opt = f"optimized_reconstruction_GD/{scene}"
os.makedirs(opt, exist_ok=True)

save_points = True # recall to set mean track len = 0 in colmap gui

epo.to_colmap(
    opt, 
    verbose=False, 
    max_points_per_image=100_000//len(epo.images), 
    save_points=save_points, 
    final_dbscan_filtering=False, 
    dbscan_eps=0.1, dbscan_min_samples=5
)

# # align to GT
# cmd = f"colmap model_aligner --input_path {opt} --output_path {opt} --ref_model_path {gt_path} --alignment_max_error 3"
# os.system(cmd)

# cmd = f"rm -rf {opt}/*.txt && colmap model_converter --input_path {opt} --output_path {opt} --output_type txt"

In [ ]:
import sys
sys.path.append('/home/mattia/Desktop/Repos/posebench/benchmarks_3D')
from benchmark_pose import eval_colmap_model

thresholds = [1,3,5]
print("AUC@",thresholds)
AUC_score_max, num_images, df_initial = eval_colmap_model(reconstruction_path, gt_path, return_df=True, thrs=thresholds)
print("VGGT AUC:   ", [float(round(_, 2)) for _ in AUC_score_max])

try:
    ba = reconstruction_path.replace("vggt", "vggt_ba")
    AUC_score_max, num_images, df_ba = eval_colmap_model(ba, gt_path, return_df=True, thrs=thresholds)
    print("VGGT+BA AUC:", [float(round(_, 2)) for _ in AUC_score_max])
    ba_ref = reconstruction_path.replace("vggt", "vggt_ba_ref")
    AUC_score_max, num_images, df_ba = eval_colmap_model(ba_ref, gt_path, return_df=True, thrs=thresholds)
    print("VGGT+BA Ref:", [float(round(_, 2)) for _ in AUC_score_max])
except:
    df_ba = None
    print("No BA reconstruction found.")
    
AUC_score_max, num_images, df_optim = eval_colmap_model(opt, gt_path, return_df=True, thrs=thresholds)
print("VGGT+EA AUC:", [float(round(_, 2)) for _ in AUC_score_max])

In [ ]:
# check cameras too
import pycolmap
ba = reconstruction_path.replace("vggt", "vggt_ba")
ba_ref = reconstruction_path.replace("vggt", "vggt_ba_ref")

!colmap model_aligner --input_path {opt} --output_path {opt} --ref_model_path {gt_path} --alignment_max_error 0.5
!colmap model_aligner --input_path {ba} --output_path {ba} --ref_model_path {gt_path} --alignment_max_error 0.5
!colmap model_aligner --input_path {ba_ref} --output_path {ba_ref} --ref_model_path {gt_path} --alignment_max_error 0.5

gt_rec = pycolmap.Reconstruction(gt_path)
vggt_rec = pycolmap.Reconstruction(reconstruction_path)
ba_rec = pycolmap.Reconstruction(ba) if df_ba is not None else None
ba_ref_rec = pycolmap.Reconstruction(ba_ref) if df_ba is not None else None
optim_rec = pycolmap.Reconstruction(opt)

In [ ]:
print("Cameras comparison:")
print("gt:", sorted([round(float(cam.params[0]), 2) for cam in gt_rec.cameras.values()]))
print("vggt:", sum(sorted([round(float(cam.params[0]), 2) for cam in vggt_rec.cameras.values()]))/len(vggt_rec.cameras))
if ba_rec is not None:
    print("ba:", sorted([round(float(cam.params[0]), 2) for cam in ba_rec.cameras.values()]))
print("edge:", sorted([round(float(cam.params[0]), 2) for cam in optim_rec.cameras.values()]))
print("alphas:", epo.intrinsics.params.cpu().detach().numpy())

In [ ]:
from modules.stopping_criterion import evaluate_R_err_fast, evaluate_t_err
# errors per camera
def compute_camera_errors(gt_rec, optim_rec):
    gt_images = [cam.name for cam in gt_rec.images.values()]

    res = {}
    for img_name in gt_images:
        gt_image = gt_rec.find_image_with_name(img_name)
        opt_image = optim_rec.find_image_with_name(img_name)

        R_gt = torch.from_numpy(gt_image.cam_from_world.rotation.matrix())
        t_gt = torch.from_numpy(gt_image.cam_from_world.translation)

        R_opt = torch.from_numpy(opt_image.cam_from_world.rotation.matrix())
        t_opt = torch.from_numpy(opt_image.cam_from_world.translation)
        dR = evaluate_R_err_fast(R_gt, R_opt).float().item()
        dt = evaluate_t_err(t_gt, t_opt).float().item()

        res[img_name] = {
            "dR": dR,
            "dt": dt
        }

    # plot sid eby side rotation and translation errors
    img_names = list(res.keys())
    dR_errors = [res[name]["dR"] for name in img_names]
    dt_errors = [res[name]["dt"] for name in img_names]
    # find distance from origin usi
    far_cameras = [(dr**2 + dt**2)**0.5 for dr, dt in zip(dR_errors, dt_errors)]
    return dR_errors, dt_errors, far_cameras


dR_errors_opt, dt_errors_opt, far_cameras_opt = compute_camera_errors(gt_rec, optim_rec)
dR_errors_ba, dt_errors_ba, far_cameras_ba = compute_camera_errors(gt_rec, ba_rec)
dR_errors_ba_ref, dt_errors_ba_ref, far_cameras_ba_ref = compute_camera_errors(gt_rec, ba_ref_rec)

n_plots = 2
plt.figure(figsize=(5,5))
plt.scatter(dR_errors_opt, dt_errors_opt, label='EA', color='red', alpha=0.5)
plt.scatter(dR_errors_ba, dt_errors_ba, label='BA', color='orange', alpha=0.5)
plt.scatter(dR_errors_ba_ref, dt_errors_ba_ref, label='BA (ref)', color='blue', alpha=0.5)
plt.xlabel("Rotation error (deg)")
plt.ylabel("Translation error (deg)")
plt.title(f"{dataset}/{scene} | Camera pose errors after optimization")
# write images names next to points above 90th percentile of far_cameras

# draw a rectagle between 0,0 and 0.5,0.5
# plt.vlines(x=2, ymin=0, ymax=2, color='g', linestyle='--')
# plt.hlines(y=2, xmin=0, xmax=2, color='g', linestyle='--')
# plt.xlim(0, max(dR_errors_opt + dR_errors_ba)*1.1)
# plt.ylim(0, max(dt_errors_opt + dt_errors_ba)*1.1)
plt.legend()
plt.show()

In [ ]:
if True:
    bins = min(50, len(df_initial)//5)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12,3))
    plt.subplot(1,2,1)
    plt.hist(df_initial.q_error, bins=bins, range=(0,15), alpha=0.5, label='initial')
    plt.hist(df_optim.q_error, bins=bins, range=(0,15), alpha=0.5, label='w/ edges')
    try:
        plt.hist(df_ba.q_error, bins=bins, range=(0,15), alpha=0.3, label='w/ ba')
    except:
        pass
    plt.title('Rotation Error')
    plt.xlabel('Error (degrees)')
    plt.ylabel('Number of pairs')
    plt.legend()
    plt.subplot(1,2,2)
    plt.hist(df_initial.t_error, bins=bins, range=(0,10), alpha=0.5, label='initial')
    plt.hist(df_optim.t_error, bins=bins, range=(0,10), alpha=0.5, label='w/ edges')
    try:
        plt.hist(df_ba.t_error, bins=bins, range=(0,10), alpha=0.3, label='w/ ba')
    except:
        pass
    plt.title('Translation Error')
    plt.xlabel('Error (degrees)')
    plt.ylabel('Number of pairs')
    plt.legend()
    plt.show()

In [ ]:
import torch
residuals_0 = torch.tensor(list(adjuster.residuals[0].values()))
residuals_final = torch.tensor(list(adjuster.residuals[-1].values()))

# print infos about residuals
print(f"Initial residuals ({len(residuals_0)}): mean={residuals_0.mean():.3f}, std={residuals_0.std():.3f}, min={residuals_0.min():.3f}, max={residuals_0.max():.3f}")
print(f"Final residuals   ({len(residuals_final)}): mean={residuals_final.mean():.3f}, std={residuals_final.std():.3f}, min={residuals_final.min():.3f}, max={residuals_final.max():.3f}")

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
out_initial = plt.hist(residuals_0, bins=100, alpha=0.7, label='initial')
# find valus for vlines
plt.vlines(residuals_0.mean(), 0, out_initial[0].max(), colors='r', linestyles='dashed', label='mean initial')
plt.title('Initial Residuals Distribution')
plt.xlabel('Residual')
plt.ylabel('Number of pairs')
plt.legend()
plt.subplot(1,2,2)
out_final = plt.hist(residuals_final, bins=100, alpha=0.7, label='final')
plt.vlines(residuals_0.mean(), 0, out_initial[0].max(), colors='r', linestyles='dashed', label='mean initial')
plt.vlines(residuals_final.mean(), 0, out_final[0].max(), colors='g', linestyles='dashed', label='mean final')
plt.title('Final Residuals Distribution')
plt.xlabel('Residual')
plt.ylabel('Number of pairs')
plt.legend()
plt.show()

In [ ]:
residuals = adjuster.compute_mre()[:, 2].astype(np.float32)
# print mean and quantiles
print(f"Mean (Edge) Reprojection Error: {residuals.mean():.4f} pixels over {len(residuals):,} pairs.")
print("Quantiles:", np.percentile(residuals, [25, 50, 75, 90]))

plt.hist(residuals, bins=100)
plt.show()

In [ ]:
import pycolmap
import numpy as np

def compute_mre_pycolmap(reconstruction):
    total_error = 0.0
    total_observations = 0
    
    # Iterate through all images in the reconstruction
    for image_id, image in reconstruction.images.items():
        # Only process registered images
        if not image.has_pose:
            continue
            
        camera = reconstruction.cameras[image.camera_id]
        
        # Get 2D-3D correspondences
        for point2D in image.points2D:
            # Check if point2D has a valid 3D point
            if not point2D.has_point3D():
                continue
            
            # Check if point3D_id is valid (not empty string)
            if not point2D.point3D_id or point2D.point3D_id not in reconstruction.points3D:
                continue
                
            # Get the 3D point coordinates
            point3D = reconstruction.points3D[point2D.point3D_id]
            
            # Project 3D point to camera frame
            # Convert Rigid3d to 4x4 matrix and apply transformation
            world_point_homo = np.append(point3D.xyz, 1.0)  # Convert to homogeneous coordinates
            cam_point_homo = image.cam_from_world.matrix() @ world_point_homo
            cam_coords = cam_point_homo[:3]  # Extract 3D coordinates
            
            # Project using camera model
            reprojected_2d = camera.img_from_cam(cam_coords)
            
            # Calculate reprojection error (L2 norm in pixels)
            observed_2d = point2D.xy
            error = np.linalg.norm(observed_2d - reprojected_2d)
            
            total_error += error
            total_observations += 1
                
    # Compute Mean Reprojection Error
    if total_observations == 0:
        return 0.0, 0
        
    mre = total_error / total_observations
    return mre, total_observations

# Usage:
for rec in [gt_rec, ba_rec, ba_ref_rec]:
    if rec is not None:
        mre_value, count = compute_mre_pycolmap(rec)
        print(f"Mean Reprojection Error: {mre_value:.4f} pixels over {count:,} observations.")

In [ ]:
.

## Reading Logs

In [ ]:
import os, json
import torch
import numpy as np
from matplotlib import pyplot as plt

def analyze_scene(logs, scene):

    # 1. CLEAN DATA
    # Ensure max_changes is a numpy array of floats
    max_changes = np.array([x.item() if torch.is_tensor(x) else x for x in logs[scene]['list_changes']['max']])
    steps = range(len(max_changes))

    th = '5'
    auc = logs[scene]['list_auc']['auc']
    raw_loss = np.array(logs[scene]['list_loss'])

    if len(max_changes) > 0:
        fig, ax1 = plt.subplots(figsize=(14, 5))

        # --- Primary Axis: Changes ---
        ax1.set_xlabel('Steps')
        ax1.set_ylabel('Max Change (Delta)', color='blue')
        
        # Plot Raw Data
        ax1.plot(steps, max_changes, alpha=0.15, label='Raw Max Change', color='blue', zorder=1)

        # Plot Smoothed Line for Window 1
        if len(max_changes) >= window1:
            smoothed1 = np.convolve(max_changes, np.ones(window1)/window1, mode='valid')
            # Offset steps to align with the end of the rolling window
            ax1.plot(steps[window1-1:], smoothed1, label=f'Smoothed (W={window1})', color='darkblue', linewidth=2, zorder=3)

        # Plot Smoothed Line for Window 2 (Visualizing what Tol2 sees)
        if len(max_changes) >= window2:
            smoothed2 = np.convolve(max_changes, np.ones(window2)/window2, mode='valid')
            ax1.plot(steps[window2-1:], smoothed2, label=f'Smoothed (W={window2})', color='teal', linewidth=1.5, linestyle='--', zorder=3)

        # Horizontal Tolerance Lines
        ax1.axhline(y=tol1, color='red', linestyle=':', alpha=0.6, label=f'Tol {tol1}')
        ax1.axhline(y=tol2, color='green', linestyle=':', alpha=0.6, label=f'Tol {tol2}')

        # --- Convergence Detection ---
        # We calculate convergence at each step based on the preceding window
        conv1 = [adjuster.check_convergence(max_changes[:i+1], window1, tol1) for i in range(len(max_changes))]
        conv2 = [adjuster.check_convergence(max_changes[:i+1], window2, tol2) for i in range(len(max_changes))]

        def plot_conv_line(conv_array, color, label_text):
            has_labeled = False
            limit_y = max(max_changes) if len(max_changes) > 0 else 1.0
            for c in range(1, len(conv_array)):
                # Detect the rising edge (False -> True)
                if conv_array[c] and not conv_array[c - 1]:
                    label = label_text if not has_labeled else None
                    ax1.vlines(steps[c], 0, limit_y, color=color, linestyle='--', linewidth=2, label=label, zorder=4)
                    has_labeled = True
                    return c

        con1_step = plot_conv_line(conv1, 'red', f'Reached Tol {tol1}')
        con2_step = plot_conv_line(conv2, 'green', f'Reached Tol {tol2}')

        # --- Loss Change Detection ---
        # loss_stable_idx = None
        # for i in range(con1_step, len(raw_loss)):
        #     prev_loss = raw_loss[i-1]
        #     curr_loss = raw_loss[i]
        #     if prev_loss != 0:
        #         rel_change = abs((curr_loss - prev_loss) / prev_loss)
        #         if rel_change < loss_change_tol:
        #             loss_stable_idx = i
        #             break
        
        # Scale loss for plotting (visual only) relative to max_changes range
        if len(raw_loss) > 0:
            # Use steps[:len(raw_loss)] to ensure X-axis alignment if lengths differ
            loss_x = steps[:len(raw_loss)] 
            l_min, l_max = raw_loss.min(), raw_loss.max()
            m_min, m_max = max_changes.min(), max_changes.max()
            
            # Avoid division by zero if loss is constant
            denom = (l_max - l_min) if l_max != l_min else 1
            scaled_loss = (raw_loss - l_min) / denom * (m_max - m_min) + m_min
            
            ax1.plot(loss_x, scaled_loss, label='Scaled Loss Trend', color='purple', alpha=0.5, linestyle=':')
        
        # # Plot Loss Stability Vertical Line
        # if loss_stable_idx is not None and loss_stable_idx < len(steps):
        #     ax1.axvline(x=steps[loss_stable_idx], color='black', linestyle='-.', linewidth=2, label='Loss Stable (<0.001%)', alpha=0.5)

        # --- Secondary Axis: AUC ---
        if th in auc and len(auc[th]) > 0:
            ax2 = ax1.twinx()
            ax2.set_ylabel('AUC', color='orange')
            
            # Correctly align AUC steps based on saving frequency
            auc_steps = np.arange(len(auc[th])) * adjuster.auc_saving_freq
            ax2.plot(auc_steps, auc[th], label='AUC (Top 5)', color='orange', marker='o', markersize=4, alpha=0.8)
            
            # Annotate AUC values
            for i, (x, y) in enumerate(zip(auc_steps, auc[th])):
                o = 0.5 if i % 2 == 0 else -2.5
                ax2.text(x, y+o, f'{y:.2f}', color='k', ha='center', va='bottom', fontsize=8, fontweight='bold')
            ax2.set_ylim(0, max(auc[th])*1.1)
            # Dynamic Y-limits for AUC
            y_min, y_max = min(auc[th]), max(auc[th])
            # ax2.set_ylim(y_min - 0.02, y_max + 0.05)

            # Merge Legends from both axes
            h1, l1 = ax1.get_legend_handles_labels()
            h2, l2 = ax2.get_legend_handles_labels()
            ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize='small', ncol=2, bbox_to_anchor=(1.05, 1))
        else:
            ax1.legend(loc='upper left', fontsize='small', ncol=2, bbox_to_anchor=(1.05, 1))

        

        ax1.set_title(f"{scene} | initial auc: {auc[th][0]:.2f} | final auc: {auc[th][round(con2_step/adjuster.auc_saving_freq)]:.2f} | max auc: {max(auc[th]):.2f}")
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

        return auc[th][round(con2_step/adjuster.auc_saving_freq)], max(auc[th]), auc[th][-1]


In [ ]:
logs = {}
scene = "room"
with open(f'optimized_reconstruction_GD/{scene}/training_logs.json', 'r') as f:
    # Use json.load() for file objects
    logs[scene] = json.load(f)
m1, m2, m3 = analyze_scene(logs, scene)

In [ ]:
window1 = 25
window2 = 50
tol1 = 0.5
tol2 = 0.1

paths = "benchmarks/vggt_edge_canny/scannetpp" # scannetpp, mipnerf360, terrasky3D

logs = {}
max_measured, max_threorethical, max_last = [], [], []
for scene in sorted(os.listdir(paths)):
    path = os.path.join(paths, scene, "sparse", "training_logs.json")
    
    # Check if the file exists to avoid FileNotFoundError
    if os.path.exists(path):
        with open(path, 'r') as f:
            # Use json.load() for file objects
            logs[scene] = json.load(f)
    
    m1, m2, m3 = analyze_scene(logs, scene)
    max_measured.append(m1)
    max_threorethical.append(m2)
    max_last.append(m3)

print(f"Scenes analyzed: {len(max_measured)}/{len(os.listdir(paths))}")
print(f"Average AUC@5 (measured): {np.mean(max_measured):.4f}")
print(f"Average AUC@5 (theoretical): {np.mean(max_threorethical):.4f}")
print(f"Average AUC@5 (last): {np.mean(max_last):.4f}")

print(f"Distance measured-theoretical: {np.mean(max_threorethical)/np.mean(max_measured)*100 - 100:.2f} %")

In [ ]:
.

## 2D Visualisation

In [ ]:
from mylib.plot import plot_imgs

viewgraph = adjuster.viewgraph
images = adjuster.images
intrinsics = adjuster.intrinsics
poses = adjuster.poses

print(len(viewgraph))

In [ ]:
# (i,j) = viewgraph[16]
i, j =  '1/_DSC8737.JPG', '1/_DSC8816.JPG' 

c = adjuster.valid_points_per_pair[(i,j)]
print(i, j, c)


plot_imgs([images[i]['image'].permute(1,2,0).cpu(), images[i]['edges_map'].cpu(),images[i]['depth'].cpu(),
           images[j]['image'].permute(1,2,0).cpu(), images[j]['edges_map'].cpu(),images[j]['depth'].cpu()], 
           titles=[f"Image {i}", "Edges", "Depth", f"Image {j}", "Edges", "Depth"],
           cmap=[None, "gray","plasma"]*2, rows=2, figsize=(12,6))

In [ ]:
x1,y1,x2,y2,h,w = [int(x) for x in images[i]['coords']]
img1 = images[i]['image'][:, y1:y2, x1:x2]
img2 = images[j]['image'][:, y1:y2, x1:x2]
Z1 = images[i]['depth'][None]  # Full depth, not cropped
Z2 = images[j]['depth'][None]  # Full depth, not cropped

In [ ]:
# Debug shapes and values
print(f"P0 shape: {poses.get_projection_matrix(i)[:1].shape}")
print(f"K0 shape: {intrinsics.get_intrinsic_matrix(images[i]['cam_id']).shape}")
print(f"Z1 shape: {Z1.shape}, min: {Z1[~torch.isnan(Z1)].min() if (~torch.isnan(Z1)).any() else 'all NaN'}")
print(f"Z2 shape: {Z2.shape}, min: {Z2[~torch.isnan(Z2)].min() if (~torch.isnan(Z2)).any() else 'all NaN'}")
print(f"img1 shape: {img1.shape}, img2 shape: {img2.shape}")

# Check valid depth percentage
valid_z1 = (~torch.isnan(Z1) & (Z1 > 0)).sum() / Z1.numel() * 100
valid_z2 = (~torch.isnan(Z2) & (Z2 > 0)).sum() / Z2.numel() * 100
print(f"Valid depth Z1: {valid_z1:.1f}%, Z2: {valid_z2:.1f}%")

In [ ]:
from helpers.reprojection import compute_121_reprojection
from mylib.plot import plot_imgs_and_kpts

data = {
    'P0': poses.get_projection_matrix(i)[:1],
    'P1': poses.get_projection_matrix(j)[:1],
    'K0': intrinsics.get_intrinsic_matrix(images[i]['cam_id']),
    'K1': intrinsics.get_intrinsic_matrix(images[j]['cam_id']),
    'depth0': Z1,
    'depth1': Z2
}

# Use same parameters as adjuster (for frustums matcher)
with torch.no_grad():
    kpts1, kpts2, tot = compute_121_reprojection(
        data,
        images[i]['image'],  # Full image
        images[j]['image'],  # Fix: use img2, not img1
        verbose=False,
        reprojection_error=3,
        border=10,
        sampling_factor=5
    )
    kpts1, kpts2 = kpts1.squeeze(), kpts2.squeeze()

print(f"Adjuster valid points: {adjuster.valid_points_per_pair.get((i,j), 'N/A')}")
print(f"Manual computation: {len(kpts1):,} out of {tot:,} ({100*len(kpts1)/tot:.1f}%)")

# Visualize on full images
plot_imgs_and_kpts(
    images[i]['image'].permute(1,2,0).cpu()*255//1,
    images[j]['image'].permute(1,2,0).cpu()*255//1,
    kpts1.detach().cpu(), kpts2.detach().cpu(),
    sample_points=-1, matches=False,
)

In [ ]:
edge1 = images[i]['edges_map'][y1:y2, x1:x2]
edge2 = images[j]['edges_map'][y1:y2, x1:x2]
edge1_kpts = edge1.nonzero().flip(dims=(0,1))
edge2_kpts = edge2.nonzero().flip(dims=(0,1))

from mylib.projections import project_points_2D_to_2D

proj_kpts1 = project_points_2D_to_2D(
    edge1_kpts,
    Z1,
    poses.get_projection_matrix(i)[:1],
    poses.get_projection_matrix(j)[:1],
    intrinsics.get_intrinsic_matrix(images[i]['cam_id']),
    intrinsics.get_intrinsic_matrix(images[j]['cam_id']),
    device="cuda"
)[1].detach().cpu()

proj_kpts2 = project_points_2D_to_2D(
    edge2_kpts,
    Z2,
    poses.get_projection_matrix(j)[:1],
    poses.get_projection_matrix(i)[:1],
    intrinsics.get_intrinsic_matrix(images[j]['cam_id']),
    intrinsics.get_intrinsic_matrix(images[i]['cam_id']),
    device="cuda"
)[1].detach().cpu()



In [ ]:
edge1 = images[i]['edges_map'][y1:y2, x1:x2]
edge2 = images[j]['edges_map'][y1:y2, x1:x2]
edge1_kpts = edge1.nonzero().flip(dims=(0,1))
edge2_kpts = edge2.nonzero().flip(dims=(0,1))

print(edge1_kpts.shape, edge2_kpts.shape)

plot_imgs_and_kpts(
    edge1[..., None].repeat(1,1,3).cpu()*255//1,
    edge2[..., None].repeat(1,1,3).cpu()*255//1,
    proj_kpts2, proj_kpts1,
    sample_points=-1, matches=False,
    point_size=5
) 

In [ ]:
from losses.dt_loss import sample_distance_field

dist = sample_distance_field(images[j]['dt_field'][None], images[i]['edges'] [None]).cpu()
print(f"Loss: {dist.mean():.4f}, dist length: {len(dist):,}, edges length: {len(images[i]['edges']):,}")

field = torch.zeros_like(images[j]['dt_field']).cpu()
edges = images[i]['edges'].long()
dist = dist.flatten()  # Ensure it's 1D

for idx, pt in enumerate(edges):
    field[pt[1].item(), pt[0].item()] = dist[idx].item()

plt.figure(figsize=(16,10))
plt.imshow(field, cmap="magma")
# plt.scatter(images[i]['edges'][:,0].cpu(), images[i]['edges'][:,1].cpu(), s=1, c='cyan')
plt.show()

In [ ]:
.